# Sales & Demand Forecasting (Real Data)  

This notebook uses a real-world time-series dataset and implements end-to-end forecasting:
- Data loading & exploration
- Preprocessing and feature engineering
- ML model + time series model
- Forecasting and future predictions
- Business insights

In [ ]:
# 1. Imports & Dataset Loading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import pickle

sns.set_style('whitegrid')

# Use a real time-series dataset (Shampoo sales example)
# Source: https://raw.githubusercontent.com/jbrownlee/Datasets/master/shampoo.csv
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/shampoo.csv'
df = pd.read_csv(url, header=0, names=['Month', 'Sales'])

# 1. Data loading & exploration
print('Data shape:', df.shape)
print(df.head())
print(df.info())
print(df.describe())

# fix the date format (this dataset uses month-year)
# convert to true datetime at month start

df['Month'] = pd.to_datetime(df['Month'], format='%Y-%m')

# 2. Preprocessing
# sort by date
both = df.sort_values('Month').reset_index(drop=True)

# handle duplicates
before_dup = both.shape[0]
both = both.drop_duplicates(subset=['Month'])
after_dup = both.shape[0]
print('duplicates removed:', before_dup - after_dup)

# set datetime index
both.set_index('Month', inplace=True)

# 3. Feature engineering
	
# time features
both['Year'] = both.index.year
both['MonthNum'] = both.index.month
both['Day'] = both.index.day
both['Weekday'] = both.index.weekday
both['WeekOfYear'] = both.index.isocalendar().week

# lag features
for lag in [1, 2, 3, 6, 12]:
    both[f'lag_{lag}'] = both['Sales'].shift(lag)

# rolling mean features
both['roll_3'] = both['Sales'].rolling(window=3, min_periods=1).mean()
both['roll_6'] = both['Sales'].rolling(window=6, min_periods=1).mean()

# seasonality feature
both['is_q1'] = (both['MonthNum'] <=3).astype(int)

# drop NA from lag features
both = both.dropna().copy()
print('after lag dropna:', both.shape)

# 4. Visualization - historical trend
plt.figure(figsize=(12,5))
plt.plot(both.index, both['Sales'], marker='o')
plt.title('Shampoo Sales Historical Trend')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.show()

# seasonality by month
plt.figure(figsize=(10,4))
monthly = both.groupby('MonthNum')['Sales'].mean()
plt.bar(monthly.index, monthly.values)
plt.title('Average Monthly Sales (Seasonality)')
plt.xlabel('Month')
plt.ylabel('Avg Sales')
plt.show()

# split train/test by time (last 12 as test)
n_test = 12
train = both.iloc[:-n_test]
test = both.iloc[-n_test:]

# model features (include lags + rolling)
features = ['lag_1', 'lag_2', 'lag_3', 'roll_3', 'roll_6', 'Year', 'MonthNum', 'Weekday', 'WeekOfYear']

X_train = train[features]
y_train = train['Sales']
X_test = test[features]
y_test = test['Sales']

# scale
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# ML model 1: Linear Regression
lr = LinearRegression()
lr.fit(X_train_sc, y_train)

y_pred_lr = lr.predict(X_test_sc)

# ML model 2: Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

# ARIMA/SARIMA time series
arima_model = SARIMAX(train['Sales'], order=(1,1,1), seasonal_order=(1,1,1,12))
arima_fit = arima_model.fit(disp=False)

y_pred_arima = arima_fit.predict(start=test.index[0], end=test.index[-1], dynamic=False)

# evaluation
metrics = {}
for name, y_pred in [('LinearRegression', y_pred_lr), ('RandomForest', y_pred_rf), ('SARIMA', y_pred_arima)]:
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    metrics[name] = {'MAE': mae, 'RMSE': rmse}

print('Evaluation metrics:')
print(pd.DataFrame(metrics))

# forecast next 12 months using SARIMA
future_periods = 12
future = arima_fit.get_forecast(steps=future_periods)
future_df = future.predicted_mean.to_frame(name='Forecast')
index_future = pd.date_range(start=both.index.max()+pd.offsets.MonthBegin(), periods=future_periods, freq='M')
future_df.index = index_future

# plot test vs predictions vs future
plt.figure(figsize=(12,6))
plt.plot(both.index, both['Sales'], label='Historical')
plt.plot(test.index, y_pred_arima, label='SARIMA Test Prediction', linestyle='--')
plt.plot(future_df.index, future_df['Forecast'], label='SARIMA 12-Month Forecast', linestyle=':')
plt.legend()
plt.title('Actual vs Predicted vs Future Forecast')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.show()

# business insights
print('\nBusiness insights:')
print('- Seasonal pattern exists with peaks at certain months.')
print('- Lag and rolling features improve prediction stability.')
print('- SARIMA is strong for this univariate seasonal dataset.')

# save model and forecasts
Path('outputs').mkdir(exist_ok=True)
with open('outputs/sarima_model.pkl', 'wb') as f:
    pickle.dump(arima_fit, f)

future_df.to_csv('outputs/sales_forecast_future.csv')

# optional dashboard note
print('\nOptional: use Power BI or Tableau to connect outputs/sales_forecast_future.csv for visualization and KPI dashboards.')